# Vision Transformer from Scratch in PyTorch

This notebook implements a compact Vision Transformer (ViT) for MNIST, following the educational structure of Matt Nguyen's Medium tutorial, **“Building a Vision Transformer Model From Scratch.”**

Source article: https://medium.com/correll-lab/building-a-vision-transformer-model-from-scratch-a3054f707cc6

The implementation below is independently written and includes practical corrections and improvements:

- vectorized sinusoidal positional encoding;
- raw logits passed to `CrossEntropyLoss` (no `Softmax` inside the model);
- dropout and configurable MLP expansion;
- explicit shape checks;
- reusable training and evaluation functions;
- patch-token and model-output inspection.


## 1. Imports and reproducibility

In [ ]:
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Configuration

For a 32×32 image and an 8×8 patch size, there are 16 patch tokens. After prepending the class token, the sequence length is 17. The embedding dimension must be divisible by the number of attention heads.


In [ ]:
@dataclass
class ViTConfig:
    image_size: int = 32
    patch_size: int = 8
    in_channels: int = 1
    num_classes: int = 10
    embed_dim: int = 96
    num_heads: int = 4
    num_layers: int = 4
    mlp_ratio: int = 4
    dropout: float = 0.1
    batch_size: int = 128
    epochs: int = 5
    learning_rate: float = 3e-4
    num_workers: int = 2

config = ViTConfig()
assert config.image_size % config.patch_size == 0
assert config.embed_dim % config.num_heads == 0
num_patches = (config.image_size // config.patch_size) ** 2
print("Number of patches:", num_patches)
print("Sequence length:", num_patches + 1)


## 3. Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = MNIST(root="./data", train=False, download=True, transform=transform)

loader_kwargs = {
    "batch_size": config.batch_size,
    "num_workers": config.num_workers,
    "pin_memory": torch.cuda.is_available(),
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

images, labels = next(iter(train_loader))
print("Images:", images.shape)
print("Labels:", labels.shape)


## 4. Patch embedding

A learnable `Conv2d` with kernel size and stride equal to the patch size extracts non-overlapping patches and projects each patch into the embedding dimension.


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError("image_size must be divisible by patch_size")
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        x = self.projection(x)  # [B, D, H/P, W/P]
        x = x.flatten(2)        # [B, D, N]
        x = x.transpose(1, 2)   # [B, N, D]
        return x

patch_embedding = PatchEmbedding(
    config.image_size,
    config.patch_size,
    config.in_channels,
    config.embed_dim,
)
with torch.no_grad():
    patch_tokens = patch_embedding(images[:4])
print("Patch tokens:", patch_tokens.shape)


## 5. Class token and sinusoidal positional encoding

In [ ]:
def build_sinusoidal_encoding(sequence_length, embed_dim):
    positions = torch.arange(sequence_length, dtype=torch.float32).unsqueeze(1)
    dimensions = torch.arange(0, embed_dim, 2, dtype=torch.float32)
    scale = torch.exp(dimensions * (-math.log(10000.0) / embed_dim))

    encoding = torch.zeros(sequence_length, embed_dim)
    encoding[:, 0::2] = torch.sin(positions * scale)
    encoding[:, 1::2] = torch.cos(
        positions * scale[:encoding[:, 1::2].shape[1]]
    )
    return encoding.unsqueeze(0)

class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, num_patches, embed_dim, dropout=0.0):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        pe = build_sinusoidal_encoding(num_patches + 1, embed_dim)
        self.register_buffer("positional_encoding", pe, persistent=False)
        self.dropout = nn.Dropout(dropout)
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, patch_tokens):
        batch_size = patch_tokens.size(0)
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, patch_tokens], dim=1)
        x = x + self.positional_encoding[:, :x.size(1)]
        return self.dropout(x)


## 6. Multi-head self-attention

The implementation computes query, key, and value projections together, splits them into heads, applies scaled dot-product attention, and merges the heads again.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim must be divisible by num_heads")
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.projection = nn.Linear(embed_dim, embed_dim)
        self.proj_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        weights = self.attn_dropout(scores.softmax(dim=-1))
        out = weights @ v
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.proj_dropout(self.projection(out))


## 7. Transformer encoder block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        hidden_dim = embed_dim * mlp_ratio
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


## 8. Complete Vision Transformer

The class-token representation is passed to a linear classifier. The model returns raw logits because `CrossEntropyLoss` already performs the required normalization internally.


In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.patch_embedding = PatchEmbedding(
            cfg.image_size, cfg.patch_size, cfg.in_channels, cfg.embed_dim
        )
        self.token_position_embedding = TokenAndPositionEmbedding(
            self.patch_embedding.num_patches, cfg.embed_dim, cfg.dropout
        )
        self.encoder = nn.Sequential(*[
            TransformerEncoderBlock(
                cfg.embed_dim,
                cfg.num_heads,
                cfg.mlp_ratio,
                cfg.dropout,
            )
            for _ in range(cfg.num_layers)
        ])
        self.final_norm = nn.LayerNorm(cfg.embed_dim)
        self.classifier = nn.Linear(cfg.embed_dim, cfg.num_classes)

    def forward_features(self, images):
        x = self.patch_embedding(images)
        x = self.token_position_embedding(x)
        x = self.encoder(x)
        x = self.final_norm(x)
        return x[:, 0]

    def forward(self, images):
        return self.classifier(self.forward_features(images))

model = VisionTransformer(config).to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
with torch.no_grad():
    logits = model(images[:4].to(device))
print("Logits:", logits.shape)


## 9. Training and evaluation functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        n = batch_labels.size(0)
        loss_sum += loss.item() * n
        correct += (logits.argmax(1) == batch_labels).sum().item()
        total += n

    return loss_sum / total, correct / total

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)

        n = batch_labels.size(0)
        loss_sum += loss.item() * n
        correct += (logits.argmax(1) == batch_labels).sum().item()
        total += n

    return loss_sum / total, correct / total


## 10. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=config.learning_rate)

history = []
for epoch in range(config.epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })
    print(
        f"Epoch {epoch + 1:02d}/{config.epochs} | "
        f"train loss {train_loss:.4f} | train acc {train_acc*100:.2f}% | "
        f"test loss {test_loss:.4f} | test acc {test_acc*100:.2f}%"
    )


## 11. Inspect predictions

In [ ]:
@torch.inference_mode()
def predict_batch(model, batch_images, device):
    model.eval()
    logits = model(batch_images.to(device))
    probabilities = logits.softmax(dim=1)
    return probabilities.argmax(dim=1).cpu(), probabilities.cpu()

sample_images, sample_labels = next(iter(test_loader))
predictions, probabilities = predict_batch(model, sample_images[:10], device)

for i in range(10):
    confidence = probabilities[i, predictions[i]].item()
    print(
        f"Sample {i:02d} | true={sample_labels[i].item()} | "
        f"predicted={predictions[i].item()} | confidence={confidence:.3f}"
    )


## 12. Save the checkpoint

In [ ]:
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": vars(config),
    "test_accuracy": test_accuracy,
}
torch.save(checkpoint, "vit_mnist_from_scratch.pt")
print(f"Saved checkpoint with test accuracy {test_accuracy*100:.2f}%")


## Architecture flow

```text
MNIST image [B, 1, 32, 32]
        ↓
Conv2D patch embedding
        ↓
Patch tokens [B, 16, 96]
        ↓
Class token + positional encoding
        ↓
Token sequence [B, 17, 96]
        ↓
Transformer encoder blocks
        ↓
Class-token representation [B, 96]
        ↓
Linear classifier
        ↓
Logits [B, 10]
```
